# AutoML Results Viewer

This notebook summarizes saved AutoML results under `data/results` with a focus on short, practical comparisons across frameworks, configurations, folds and final selections.


In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)

METRIC_PRIORITY = ["rmse", "mae", "mse", "r2", "accuracy", "f1", "roc_auc", "auc", "precision", "recall"]
LOWER_IS_BETTER = {"rmse", "mae", "mse", "logloss", "loss", "error"}
SIMPLE_MODEL_RANK = {
    "GLM": 1,
    "LinearRegression": 1,
    "RidgeCV": 1,
    "LassoLarsCV": 1,
    "DecisionTreeRegressor": 2,
    "DRF": 3,
    "RandomForest": 3,
    "GBM": 4,
    "XGBoost": 5,
    "DeepLearning": 5,
    "Pipeline": 5,
    "StackedEnsemble": 6,
}


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "results").exists():
            return candidate
        if (candidate / "dengue_prediction" / "data" / "results").exists():
            return candidate / "dengue_prediction"
    raise FileNotFoundError("Could not locate the project root containing data/results.")


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
MODELS_DIR = RESULTS_DIR / "models" / "autoML"
REPORTS_DIR = RESULTS_DIR / "autoML"


def load_json(path: Path) -> dict | list | None:
    try:
        with open(path, "r", encoding="utf-8") as file_obj:
            return json.load(file_obj)
    except Exception:
        return None


def safe_float(value):
    try:
        if value is None:
            return None
        value = float(value)
        if math.isnan(value):
            return None
        return value
    except Exception:
        return None


def shorten(value, width: int = 72):
    if value is None:
        return None
    text = str(value).replace("\n", " ")
    return text if len(text) <= width else text[: width - 3] + "..."


def stage_sort_key(stage: str):
    stage = str(stage)
    if stage.startswith("fold_"):
        try:
            return (0, int(stage.split("_")[-1]))
        except Exception:
            return (0, 999)
    if stage == "final":
        return (1, 999)
    return (2, 999)


def first_metric(metrics: dict | None):
    metrics = metrics or {}
    for key in METRIC_PRIORITY:
        if key in metrics and metrics.get(key) is not None:
            return key, safe_float(metrics.get(key)), "lower" if key in LOWER_IS_BETTER else "higher"
    for key, value in metrics.items():
        metric_value = safe_float(value)
        if metric_value is not None:
            direction = "lower" if any(token in key.lower() for token in LOWER_IS_BETTER) else "higher"
            return key, metric_value, direction
    return None, None, None


def record_metric(stage: str, metadata: dict):
    metrics = metadata.get("metrics") or {}
    if stage == "final":
        final_metric_priority = [
            "mean_rmse",
            "mean_mae",
            "mean_mse",
            "mean_r2",
            "mean_accuracy",
            "mean_f1",
            "mean_roc_auc",
            "mean_auc",
            "mean_precision",
            "mean_recall",
        ]
        for key in final_metric_priority:
            if key in metrics and metrics.get(key) is not None:
                base_key = key.replace("mean_", "")
                direction = "lower" if base_key in LOWER_IS_BETTER else "higher"
                return key, safe_float(metrics.get(key)), direction
        validation_score = safe_float(metadata.get("validation_score"))
        if validation_score is not None:
            return "validation_score", validation_score, "lower"
    return first_metric(metrics)


def format_metric(metric_name: str | None, metric_value: float | None):
    if metric_name is None or metric_value is None:
        return "not available"
    return f"{metric_name}={metric_value:.4f}"


def format_seconds(value):
    seconds = safe_float(value)
    if seconds is None:
        return None
    return round(seconds, 3)


def report_path(config: str, framework: str) -> Path | None:
    candidate = REPORTS_DIR / config / f"{framework}_report.json"
    return candidate if candidate.exists() else None


def pipeline_hint(metadata: dict):
    pipeline_steps = metadata.get("pipeline_steps") or {}
    all_steps = pipeline_steps.get("all_steps") or []
    final_estimator = (pipeline_steps.get("final_estimator") or {}).get("class_name")
    if all_steps:
        return len(all_steps), f"{len(all_steps)} steps | {final_estimator or metadata.get('model_type') or 'unknown'}"
    model_type = metadata.get("algo") or metadata.get("model_type")
    return SIMPLE_MODEL_RANK.get(str(model_type)), str(model_type) if model_type else "not available"


def collect_run_records() -> pd.DataFrame:
    rows = []
    if not MODELS_DIR.exists():
        return pd.DataFrame()

    for metadata_path in sorted(MODELS_DIR.glob("*/*/*/metadata.json")):
        try:
            config, framework, stage, _ = metadata_path.relative_to(MODELS_DIR).parts
        except ValueError:
            continue

        metadata = load_json(metadata_path) or {}
        metrics = metadata.get("metrics") or {}
        metric_name, metric_value, metric_direction = record_metric(stage, metadata)
        complexity_score, complexity_label = pipeline_hint(metadata)
        available_metrics = [key for key, value in metrics.items() if value is not None]

        rows.append(
            {
                "framework": framework,
                "config": config,
                "stage": stage,
                "best_model": metadata.get("model_name") or metadata.get("model_id"),
                "model_type": metadata.get("model_type") or metadata.get("algo"),
                "training_time": safe_float(metadata.get("training_time")),
                "validation_score": safe_float(metadata.get("validation_score")),
                "metrics": metrics,
                "metrics_available": ", ".join(available_metrics) if available_metrics else "not available",
                "metric_name": metric_name,
                "metric_value": metric_value,
                "metric_direction": metric_direction,
                "metric_summary": format_metric(metric_name, metric_value),
                "artifact_path": metadata.get("model_path"),
                "metadata_path": str(metadata_path),
                "leaderboard_path": metadata.get("leaderboard_path"),
                "history_path": str(report_path(config, framework)) if report_path(config, framework) else None,
                "complexity_score": complexity_score,
                "complexity_proxy": complexity_label,
            }
        )

    runs_df = pd.DataFrame(rows)
    if runs_df.empty:
        return runs_df

    return runs_df.sort_values(by=["framework", "config", "stage"], key=lambda s: s.map(stage_sort_key) if s.name == "stage" else s).reset_index(drop=True)


def collect_history(config: str, framework: str, top_n: int = 5) -> pd.DataFrame:
    path = report_path(config, framework)
    report = load_json(path) if path else None
    history = (report or {}).get("model_history") or []
    if not history:
        return pd.DataFrame()

    history_df = pd.DataFrame(history).head(top_n).copy()
    if "model_name" in history_df.columns:
        history_df["model_name"] = history_df["model_name"].map(lambda value: shorten(value, 60))
    history_df.insert(0, "config", config)
    columns = [
        "config",
        "rank",
        "model_family",
        "validation_score",
        "fit_time",
        "status",
        "selected_in_final_ensemble",
        "generation_or_iteration",
    ]
    available_columns = [column for column in columns if column in history_df.columns]
    return history_df[available_columns]


def fold_consistency(group: pd.DataFrame):
    fold_df = group[group["stage"].str.startswith("fold_")].copy()
    metric_names = fold_df["metric_name"].dropna().unique().tolist()
    if len(metric_names) != 1:
        return None
    values = fold_df["metric_value"].dropna()
    if len(values) < 2:
        return None
    return float(values.std())


runs_df = collect_run_records()
display(Markdown(f"`project root:` {PROJECT_ROOT}"))


`project root:` C:\DEV\dengue_prediction\dengue_prediction

## Overview


In [2]:
if runs_df.empty:
    display(Markdown("No AutoML model metadata was found under `data/results/models/autoML`."))
else:
    frameworks = sorted(runs_df["framework"].unique().tolist())
    configs = sorted(runs_df["config"].unique().tolist())
    total_models = int(runs_df["artifact_path"].notna().sum())
    final_models = int((runs_df["stage"] == "final").sum())

    overview = pd.DataFrame(
        [
            {"item": "frameworks found", "value": ", ".join(frameworks)},
            {"item": "configurations found", "value": ", ".join(configs)},
            {"item": "total model records found", "value": total_models},
            {"item": "final model records", "value": final_models},
        ]
    )
    display(overview)

    framework_overview = (
        runs_df.groupby("framework")
        .agg(configs=("config", lambda values: ", ".join(sorted(set(values)))), runs=("stage", "count"))
        .reset_index()
    )
    display(framework_overview)


,item,value
0,frameworks found,"h2o, tpot"
1,configurations found,"high, low, medium"
2,total model records found,30
3,final model records,5


,framework,configs,runs
0,h2o,"high, low, medium",18
1,tpot,"low, medium",12


# APRESENTACAO

| Biblioteca | Abordagem | Saída | Observação |
|---|---|---|---|
| TPOT | Algoritmo genético | Pipeline sklearn | Mais interpretável |
| H2O AutoML | Vários modelos + ranking | Melhor modelo + leaderboard | Robusto, exige Java |
| AutoSklearn | Meta-learning + otimização bayesiana | Modelo/ensemble | Difícil no Windows |

## Data set - Dengue em Recife
- Problema: regressão
- Variável alvo: `casos_dengue`
- Métrica usada: 
  - MAE: Erro Absoluto Médio
  - MSE: Erro Quadrático Médio
  - RMSE: Raiz do Erro Quadrático Médio
  - R²: Coeficiente de Determinação

## 🧬 TPOT

parâmetros principais:
| Parâmetro | O que significa |
|---|---|
| `generations` | Número de gerações |
| `population_size` | Quantidade de pipelines candidatas em cada geração |
| `cv` | Número de divisões usadas na validação interna |
| `early_stop` | Para a busca se não houver melhora após algumas gerações |
| `max_eval_time_mins` | Tempo máximo para avaliar uma pipeline |
| `n_jobs` | Número de núcleos usados em paralelo |
| `processes` | Quantidade de processos paralelos |

```python

params = {
        "generations": 5,
        "population_size": 50,
        "scorers": ["neg_mean_squared_error"],
        "cv": 5,
        "random_state": 42,
        "verbosity": 2,
        "n_jobs": -1,
    }
model = TPOTRegressor(**params)
model.fit(X, y)
y_pred = model.predict(X_test)
```

## H2O AutoML: 

parâmetros principais:
| Parâmetro | O que significa |
|---|---|
| `max_models` | Número máximo de modelos treinados |
| `max_runtime_secs` | Tempo máximo total da busca |
| `max_runtime_secs_per_model` | Tempo máximo para cada modelo individual |
| `nfolds` | Número de folds internos usados pelo H2O |
| `stopping_metric` | Métrica usada para decidir parada |
| `sort_metric` | Métrica usada para ordenar o leaderboard |

## AutoSklearn

###  Ideia Principal

- Usa:
  - Meta-learning
  - Otimização bayesiana
  - Ensembles automáticos

## Framework Summaries


In [3]:
if runs_df.empty:
    display(Markdown("No framework summaries are available."))
else:
    for framework in sorted(runs_df["framework"].unique()):
        framework_df = runs_df[runs_df["framework"] == framework].copy()
        framework_df = framework_df.sort_values(by=["config", "stage"], key=lambda s: s.map(stage_sort_key) if s.name == "stage" else s)
        config_list = ", ".join(sorted(framework_df["config"].unique().tolist()))
        fold_list = ", ".join(sorted([stage for stage in framework_df["stage"].unique().tolist() if str(stage).startswith("fold_")], key=stage_sort_key))

        display(Markdown(f"### {framework}"))
        display(Markdown(f"`configs:` {config_list}  \n`folds:` {fold_list or 'not available'}"))

        summary_table = framework_df[[
            "config",
            "stage",
            "best_model",
            "model_type",
            "metric_summary",
            "metrics_available",
            "training_time",
            "artifact_path",
        ]].copy()
        summary_table["best_model"] = summary_table["best_model"].map(lambda value: shorten(value, 80))
        summary_table["training_time"] = summary_table["training_time"].map(format_seconds)
        summary_table = summary_table.rename(columns={"stage": "run", "training_time": "train_s"})
        display(summary_table)

        history_tables = []
        for config in sorted(framework_df["config"].unique()):
            history_df = collect_history(config, framework, top_n=5)
            if not history_df.empty:
                history_tables.append(history_df)

        if history_tables:
            display(Markdown("`top history rows per config:`"))
            display(pd.concat(history_tables, ignore_index=True))
        else:
            display(Markdown("`top history rows per config:` not available"))


### h2o

`configs:` high, low, medium  
`folds:` fold_1, fold_2, fold_3, fold_4, fold_5

,config,run,best_model,model_type,metric_summary,metrics_available,train_s,artifact_path
0,high,fold_1,GBM_grid_1_AutoML_1_20260425_151645_model_1,GBM,rmse=114.0819,"mae, mse, rmse, r2",10.083,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\high\h2o\fold_1\GBM_grid_1_AutoML_1_20260425_1...
1,high,fold_2,DRF_1_AutoML_2_20260425_151655,DRF,rmse=85.6424,"mae, mse, rmse, r2",11.840,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\high\h2o\fold_2\DRF_1_AutoML_2_20260425_151655
2,high,fold_3,GBM_4_AutoML_3_20260425_151708,GBM,rmse=12.3715,"mae, mse, rmse, r2",11.992,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\high\h2o\fold_3\GBM_4_AutoML_3_20260425_151708
3,high,fold_4,StackedEnsemble_AllModels_1_AutoML_4_20260425_151721,StackedEnsemble,rmse=20.9325,"mae, mse, rmse, r2",16.000,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\high\h2o\fold_4\StackedEnsemble_AllModels_1_Au...
4,high,fold_5,GBM_grid_1_AutoML_5_20260425_151737_model_1,GBM,rmse=36.0788,"mae, mse, rmse, r2",17.609,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\high\h2o\fold_5\GBM_grid_1_AutoML_5_20260425_1...
5,high,final,StackedEnsemble_AllModels_1_AutoML_6_20260425_151756,StackedEnsemble,mean_rmse=53.8214,"fold_count, fold_metrics, mean_mae, std_mae, mean_mse, std_mse, mean_r2, std_r2, mean_rmse, std_rmse",17.587,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\high\h2o\final\StackedEnsemble_AllModels_1_Aut...
6,low,fold_1,GBM_1_AutoML_1_20260425_150548,GBM,rmse=117.0761,"mae, mse, rmse, r2",1.249,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\h2o\fold_1\GBM_1_AutoML_1_20260425_150548
7,low,fold_2,GBM_1_AutoML_2_20260425_150550,GBM,rmse=105.1539,"mae, mse, rmse, r2",0.621,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\h2o\fold_2\GBM_1_AutoML_2_20260425_150550
8,low,fold_3,GBM_1_AutoML_3_20260425_150552,GBM,rmse=13.1141,"mae, mse, rmse, r2",0.836,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\h2o\fold_3\GBM_1_AutoML_3_20260425_150552
9,low,fold_4,GBM_1_AutoML_4_20260425_150554,GBM,rmse=20.7365,"mae, mse, rmse, r2",0.831,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\h2o\fold_4\GBM_1_AutoML_4_20260425_150554


`top history rows per config:`

,config,rank,model_family,validation_score,fit_time,status,selected_in_final_ensemble,generation_or_iteration
0,high,1,StackedEnsemble,32.449182,0.333,ok,True,1
1,high,2,StackedEnsemble,32.498979,0.269,ok,False,2
2,high,3,GBM,32.675557,0.179,ok,False,3
3,high,4,DRF,32.756909,0.815,ok,False,4
4,high,5,DRF,32.891434,0.500,ok,False,5
5,low,1,GBM,28.724809,0.117,ok,True,1
6,low,2,GLM,39.221080,0.041,ok,False,2
7,medium,1,GBM,28.222975,0.258,ok,True,1
8,medium,2,GBM,28.629325,0.109,ok,False,2
9,medium,3,GBM,28.816460,0.154,ok,False,3


### tpot

`configs:` low, medium  
`folds:` fold_1, fold_2, fold_3, fold_4, fold_5

,config,run,best_model,model_type,metric_summary,metrics_available,train_s,artifact_path
18,low,fold_1,Pipeline,Pipeline,rmse=117.7323,"mae, mse, rmse, r2",5.022,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\tpot\fold_1\model_20260425T175209178527Z_1...
19,low,fold_2,Pipeline,Pipeline,rmse=113.2148,"mae, mse, rmse, r2",9.284,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\tpot\fold_2\model_20260425T175218613877Z_1...
20,low,fold_3,Pipeline,Pipeline,rmse=122.9327,"mae, mse, rmse, r2",13.479,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\tpot\fold_3\model_20260425T175232268996Z_1...
21,low,fold_4,Pipeline,Pipeline,rmse=91.8339,"mae, mse, rmse, r2",10.340,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\tpot\fold_4\model_20260425T175242814729Z_1...
22,low,fold_5,Pipeline,Pipeline,rmse=48.4681,"mae, mse, rmse, r2",13.423,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\tpot\fold_5\model_20260425T175256360218Z_1...
23,low,final,Pipeline,Pipeline,mean_rmse=98.8364,"fold_count, fold_metrics, mean_mae, std_mae, mean_mse, std_mse, mean_r2, std_r2, mean_rmse, std_rmse",19.041,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\tpot\final\model_20260425T175316060288Z_1....
24,medium,fold_1,Pipeline,Pipeline,rmse=116.8880,"mae, mse, rmse, r2",23.612,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\medium\tpot\fold_1\model.joblib
25,medium,fold_2,Pipeline,Pipeline,rmse=73.3843,"mae, mse, rmse, r2",206.776,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\medium\tpot\fold_2\model.joblib
26,medium,fold_3,Pipeline,Pipeline,rmse=63.3832,"mae, mse, rmse, r2",466.694,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\medium\tpot\fold_3\model.joblib
27,medium,fold_4,Pipeline,Pipeline,rmse=43.5360,"mae, mse, rmse, r2",879.515,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\medium\tpot\fold_4\model.joblib


`top history rows per config:`

,config,rank,model_family,validation_score,fit_time,status,selected_in_final_ensemble,generation_or_iteration
0,low,1,Pipeline,None,0.0,failed,False,0.0
1,low,2,Pipeline,None,0.0,ok,True,0.0
2,low,3,Pipeline,None,0.0,failed,False,0.0
3,low,4,Pipeline,None,0.0,ok,False,0.0
4,low,5,Pipeline,None,0.0,ok,False,0.0
5,medium,1,Pipeline,None,0.0,failed,False,0.0
6,medium,2,Pipeline,None,0.0,ok,False,0.0
7,medium,3,Pipeline,None,0.0,failed,False,0.0
8,medium,4,Pipeline,None,0.0,ok,False,0.0
9,medium,5,Pipeline,None,0.0,ok,False,0.0


## Final Comparative Summary


In [4]:
if runs_df.empty:
    display(Markdown("No final comparison is available."))
else:
    consistency_rows = []
    for (framework, config), group in runs_df.groupby(["framework", "config"]):
        consistency_rows.append(
            {
                "framework": framework,
                "config": config,
                "fold_metric_std": fold_consistency(group),
            }
        )
    consistency_df = pd.DataFrame(consistency_rows)

    final_df = runs_df[runs_df["stage"] == "final"].copy()
    final_df = final_df.merge(consistency_df, on=["framework", "config"], how="left")
    final_df["training_time"] = final_df["training_time"].map(format_seconds)
    final_df["best_model"] = final_df["best_model"].map(lambda value: shorten(value, 80))

    comparison_table = final_df[[
        "framework",
        "config",
        "best_model",
        "model_type",
        "metric_name",
        "metric_value",
        "training_time",
        "complexity_proxy",
        "fold_metric_std",
        "artifact_path",
    ]].rename(columns={"training_time": "train_s", "fold_metric_std": "fold_std", "complexity_proxy": "simplicity_proxy"})
    display(comparison_table)

    answer_rows = []
    comparable_metric_names = [name for name in final_df["metric_name"].dropna().unique().tolist()]
    if len(comparable_metric_names) == 1:
        metric_name = comparable_metric_names[0]
        comparable_df = final_df[final_df["metric_name"] == metric_name].dropna(subset=["metric_value"]).copy()
        if not comparable_df.empty:
            direction = comparable_df["metric_direction"].dropna().iloc[0]
            best_idx = comparable_df["metric_value"].idxmin() if direction == "lower" else comparable_df["metric_value"].idxmax()
            best_row = comparable_df.loc[best_idx]
            answer_rows.append({
                "question": "Which AutoML library performed best?",
                "answer": f"{best_row['framework']} ({best_row['config']})",
                "basis": format_metric(best_row['metric_name'], best_row['metric_value']),
            })
            answer_rows.append({
                "question": "Which configuration gave the best result?",
                "answer": f"{best_row['config']} via {best_row['framework']}",
                "basis": format_metric(best_row['metric_name'], best_row['metric_value']),
            })
        else:
            answer_rows.append({"question": "Which AutoML library performed best?", "answer": "not available", "basis": "metric values missing"})
            answer_rows.append({"question": "Which configuration gave the best result?", "answer": "not available", "basis": "metric values missing"})
    else:
        answer_rows.append({"question": "Which AutoML library performed best?", "answer": "not directly comparable", "basis": ", ".join(comparable_metric_names) or "no shared metric"})
        answer_rows.append({"question": "Which configuration gave the best result?", "answer": "not directly comparable", "basis": ", ".join(comparable_metric_names) or "no shared metric"})

    training_df = final_df.dropna(subset=["training_time"]).copy()
    if not training_df.empty:
        fastest_row = training_df.loc[training_df["training_time"].idxmin()]
        answer_rows.append({
            "question": "Which one was faster?",
            "answer": f"{fastest_row['framework']} ({fastest_row['config']})",
            "basis": f"train_s={fastest_row['training_time']}",
        })
    else:
        answer_rows.append({"question": "Which one was faster?", "answer": "not available", "basis": "training time missing"})

    simple_df = final_df.dropna(subset=["complexity_score"]).copy()
    if not simple_df.empty:
        simplest_row = simple_df.loc[simple_df["complexity_score"].idxmin()]
        answer_rows.append({
            "question": "Which model seems lighter/simpler?",
            "answer": f"{simplest_row['framework']} ({simplest_row['config']})",
            "basis": simplest_row['complexity_proxy'],
        })
    else:
        answer_rows.append({"question": "Which model seems lighter/simpler?", "answer": "not available", "basis": "no simplicity proxy found"})

    consistency_only_df = final_df.dropna(subset=["fold_metric_std", "metric_name"]).copy()
    if not consistency_only_df.empty:
        most_consistent_row = consistency_only_df.loc[consistency_only_df["fold_metric_std"].idxmin()]
        answer_rows.append({
            "question": "Which one looks most consistent across folds?",
            "answer": f"{most_consistent_row['framework']} ({most_consistent_row['config']})",
            "basis": f"std({most_consistent_row['metric_name']})={most_consistent_row['fold_metric_std']:.4f}",
        })
    else:
        answer_rows.append({"question": "Which one looks most consistent across folds?", "answer": "not available", "basis": "insufficient fold metrics"})

    answers_df = pd.DataFrame(answer_rows)
    display(answers_df)


,framework,config,best_model,model_type,metric_name,metric_value,train_s,simplicity_proxy,fold_std,artifact_path
0,h2o,high,StackedEnsemble_AllModels_1_AutoML_6_20260425_151756,StackedEnsemble,mean_rmse,53.821410,17.587,StackedEnsemble,44.041129,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\high\h2o\final\StackedEnsemble_AllModels_1_Aut...
1,h2o,low,GBM_1_AutoML_6_20260425_150556,GBM,mean_rmse,58.772290,0.408,GBM,48.791992,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\h2o\final\GBM_1_AutoML_6_20260425_150556
2,h2o,medium,GBM_1_AutoML_6_20260425_151602,GBM,mean_rmse,53.664793,1.418,GBM,43.404595,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\medium\h2o\final\GBM_1_AutoML_6_20260425_151602
3,tpot,low,Pipeline,Pipeline,mean_rmse,98.836377,19.041,2 steps | Pipeline,30.538635,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\low\tpot\final\model_20260425T175316060288Z_1....
4,tpot,medium,Pipeline,Pipeline,mean_rmse,69.698362,2558.687,2 steps | Pipeline,28.737114,C:\DEV\dengue_prediction\dengue_prediction\data\results\models\autoML\medium\tpot\final\model.joblib


,question,answer,basis
0,Which AutoML library performed best?,h2o (medium),mean_rmse=53.6648
1,Which configuration gave the best result?,medium via h2o,mean_rmse=53.6648
2,Which one was faster?,h2o (low),train_s=0.408
3,Which model seems lighter/simpler?,tpot (low),2 steps | Pipeline
4,Which one looks most consistent across folds?,tpot (medium),std(mean_rmse)=28.7371


# Best Models Detailed Analysis


In [5]:
if runs_df.empty or "final_df" not in globals() or final_df.empty:
    display(Markdown("No detailed model analysis is available."))
else:
    import os
    import pickle
    import tempfile

    def compact_value(value, width: int = 28):
        if value is None:
            return None
        text = str(value).replace("\n", " ")
        return text if len(text) <= width else text[: width - 3] + "..."


    def compact_error(error, width: int = 96):
        if not error:
            return "not available"
        text = str(error).splitlines()[0].strip()
        return compact_value(text, width) or "not available"


    def comparison_metric_text(row):
        parts = []
        metric_name = row.get("metric_name")
        metric_value = row.get("metric_value")
        if metric_name and pd.notna(metric_value):
            parts.append(format_metric(metric_name, metric_value))
        validation_score = row.get("validation_score")
        if pd.notna(validation_score) and metric_name != "validation_score":
            parts.append(f"validation_score={validation_score:.4f}")
        return "; ".join(parts) if parts else "not available"


    def collect_winning_rows(final_df: pd.DataFrame):
        winners = []
        comparable_metric_names = [name for name in final_df["metric_name"].dropna().unique().tolist()]
        if len(comparable_metric_names) == 1:
            comparable_df = final_df[final_df["metric_name"] == comparable_metric_names[0]].dropna(subset=["metric_value"]).copy()
            if not comparable_df.empty:
                direction = comparable_df["metric_direction"].dropna().iloc[0]
                best_idx = comparable_df["metric_value"].idxmin() if direction == "lower" else comparable_df["metric_value"].idxmax()
                winners.append(("best overall performance", comparable_df.loc[best_idx]))
                winners.append(("best configuration", comparable_df.loc[best_idx]))

        training_df = final_df.dropna(subset=["training_time"]).copy()
        if not training_df.empty:
            winners.append(("fastest training time", training_df.loc[training_df["training_time"].idxmin()]))

        simple_df = final_df.dropna(subset=["complexity_score"]).copy()
        if not simple_df.empty:
            winners.append(("lightest/simplest model", simple_df.loc[simple_df["complexity_score"].idxmin()]))

        consistency_df = final_df.dropna(subset=["fold_metric_std", "metric_name"]).copy()
        if not consistency_df.empty:
            winners.append(("most stable across folds", consistency_df.loc[consistency_df["fold_metric_std"].idxmin()]))
        return winners


    def unique_winners(winners):
        unique = {}
        for reason, row in winners:
            artifact_path = row.get("artifact_path")
            key = str(artifact_path) if artifact_path else str(row.get("metadata_path"))
            if key not in unique:
                unique[key] = {"row": row.copy(), "reasons": []}
            if reason not in unique[key]["reasons"]:
                unique[key]["reasons"].append(reason)
        return unique


    def load_metadata_from_row(row):
        metadata_path = Path(str(row["metadata_path"]))
        return load_json(metadata_path) or {}


    def load_sklearn_like_model(path: Path):
        try:
            import joblib

            return joblib.load(path), None
        except Exception as joblib_error:
            try:
                with open(path, "rb") as file_obj:
                    return pickle.load(file_obj), None
            except Exception as pickle_error:
                return None, f"joblib: {joblib_error} | pickle: {pickle_error}"


    def ensure_h2o_connection_for_notebook():
        import h2o

        try:
            if h2o.connection() is not None:
                return h2o
        except Exception:
            pass

        ice_root = PROJECT_ROOT / "temp" / "h2o_notebook_runtime"
        ice_root.mkdir(parents=True, exist_ok=True)
        for env_key in ("TMP", "TEMP", "TMPDIR"):
            os.environ[env_key] = str(ice_root)
        tempfile.tempdir = str(ice_root)
        h2o.init(max_mem_size="1G", nthreads=1, verbose=False, ice_root=str(ice_root), log_dir=str(ice_root))
        return h2o


    def load_h2o_model_artifact(path: Path):
        try:
            h2o = ensure_h2o_connection_for_notebook()
            return h2o.load_model(str(path)), None
        except Exception as error:
            return None, str(error)


    def load_model_once(row, metadata, cache):
        artifact_path = metadata.get("model_path") or row.get("artifact_path")
        cache_key = str(artifact_path) if artifact_path else str(row.get("metadata_path"))
        if cache_key in cache:
            return cache[cache_key]

        framework = str(row.get("framework", "")).lower()
        path = Path(str(artifact_path)) if artifact_path else None
        loaded_model, error = None, None

        if path and path.exists():
            if framework == "h2o":
                loaded_model, error = load_h2o_model_artifact(path)
            else:
                loaded_model, error = load_sklearn_like_model(path)
        else:
            error = f"artifact not found: {artifact_path}"

        cache[cache_key] = {"artifact_path": artifact_path, "model": loaded_model, "error": error}
        return cache[cache_key]


    def format_param_pairs(params, preferred=None, limit: int = 4):
        params = params or {}
        preferred = preferred or []
        ordered_items = []
        for key in preferred:
            if key in params and params[key] is not None:
                ordered_items.append((key, params[key]))
        for key, value in params.items():
            if key in preferred or value is None:
                continue
            ordered_items.append((key, value))
        if not ordered_items:
            return "not available"
        return ", ".join(f"{key}={compact_value(value)}" for key, value in ordered_items[:limit])


    def filtered_sklearn_params(estimator):
        ignored = {
            "memory",
            "verbose",
            "n_jobs",
            "steps",
            "transformer_list",
            "transformers",
            "remainder",
            "sparse_threshold",
            "transform_input",
            "feature_names_out",
            "verbose_feature_names_out",
        }
        try:
            params = estimator.get_params(deep=False)
        except Exception:
            return {}

        defaults = {}
        try:
            defaults = estimator.__class__().get_params(deep=False)
        except Exception:
            defaults = {}

        selected = {}
        for key, value in params.items():
            if key in ignored or value is None:
                continue
            if key in defaults and defaults.get(key) == value:
                continue
            if isinstance(value, (list, tuple, dict, set)):
                continue
            selected[key] = value

        if not selected:
            fallback_keys = [
                "criterion",
                "alpha",
                "C",
                "kernel",
                "degree",
                "ntrees",
                "max_depth",
                "learn_rate",
                "percentile",
                "quantile_range",
                "min_samples_leaf",
                "min_samples_split",
            ]
            for key in fallback_keys:
                value = params.get(key)
                if key not in ignored and value is not None:
                    selected[key] = value
        return selected


    def collect_pipeline_rows(estimator):
        rows = []

        def visit_step(step, prefix, is_final=False):
            has_children = hasattr(step, "steps") or hasattr(step, "transformer_list") or hasattr(step, "transformers")
            estimator_label = step.__class__.__name__
            if is_final and not has_children:
                estimator_label = f"{estimator_label} [final]"
            rows.append(
                {
                    "step": prefix,
                    "estimator": estimator_label,
                    "key hyperparameters": format_param_pairs(filtered_sklearn_params(step)),
                }
            )
            if hasattr(step, "steps"):
                nested_steps = list(step.steps)
                for index, (child_name, child_step) in enumerate(nested_steps, start=1):
                    visit_step(child_step, f"{prefix} > {index}.{child_name}", is_final=is_final and index == len(nested_steps))
            elif hasattr(step, "transformer_list"):
                nested_steps = list(step.transformer_list)
                for index, (child_name, child_step) in enumerate(nested_steps, start=1):
                    if child_step in (None, "drop", "passthrough"):
                        continue
                    visit_step(child_step, f"{prefix} > {index}.{child_name}", is_final=False)
            elif hasattr(step, "transformers"):
                nested_steps = list(step.transformers)
                for index, transformer in enumerate(nested_steps, start=1):
                    if len(transformer) < 2:
                        continue
                    child_name, child_step = transformer[0], transformer[1]
                    if child_step in (None, "drop", "passthrough"):
                        continue
                    visit_step(child_step, f"{prefix} > {index}.{child_name}", is_final=False)

        if hasattr(estimator, "steps"):
            top_steps = list(estimator.steps)
            for index, (name, step) in enumerate(top_steps, start=1):
                visit_step(step, f"{index}.{name}", is_final=index == len(top_steps))
        else:
            rows.append(
                {
                    "step": "1.model",
                    "estimator": f"{estimator.__class__.__name__} [final]",
                    "key hyperparameters": format_param_pairs(filtered_sklearn_params(estimator)),
                }
            )

        if len(rows) > 8:
            final_row = next((row for row in reversed(rows) if "[final]" in str(row.get("estimator", ""))), rows[-1])
            trimmed_rows = rows[:6]
            if final_row not in trimmed_rows:
                trimmed_rows.append({"step": "...", "estimator": "truncated", "key hyperparameters": "intermediate nested steps omitted"})
                trimmed_rows.append(final_row)
            rows = trimmed_rows
        return pd.DataFrame(rows)


    def extract_h2o_spec(loaded_model, metadata):
        algo = metadata.get("algo") or metadata.get("model_type") or metadata.get("model_name")
        key_params = {}
        priority = [
            "ntrees",
            "max_depth",
            "learn_rate",
            "sample_rate",
            "col_sample_rate",
            "col_sample_rate_per_tree",
            "min_rows",
            "mtries",
            "distribution",
            "alpha",
            "lambda",
            "stopping_rounds",
            "stopping_metric",
            "stopping_tolerance",
        ]

        params = getattr(loaded_model, "params", {}) if loaded_model is not None else {}
        for key in priority:
            value = params.get(key)
            if isinstance(value, dict):
                actual = value.get("actual")
                default = value.get("default")
                if actual not in (None, "AUTO") and actual != default:
                    key_params[key] = actual
            elif value not in (None, "AUTO"):
                key_params[key] = value

        return pd.DataFrame(
            [
                {
                    "algorithm": algo or "not available",
                    "key hyperparameters": format_param_pairs(key_params, preferred=priority, limit=5),
                }
            ]
        )


    def extract_autosklearn_spec(loaded_model):
        if loaded_model is None:
            return pd.DataFrame()
        if hasattr(loaded_model, "show_models"):
            try:
                model_text = str(loaded_model.show_models()).splitlines()[0]
            except Exception:
                model_text = loaded_model.__class__.__name__
            rows = [{"component": "ensemble", "estimator": compact_value(model_text, 72), "key hyperparameters": "not available"}]
            final_estimator = getattr(loaded_model, "automl_", None)
            if final_estimator is not None:
                rows.append(
                    {
                        "component": "final estimator",
                        "estimator": final_estimator.__class__.__name__,
                        "key hyperparameters": format_param_pairs(filtered_sklearn_params(final_estimator)),
                    }
                )
            return pd.DataFrame(rows)
        return collect_pipeline_rows(loaded_model)


    winners = unique_winners(collect_winning_rows(final_df))
    if not winners:
        display(Markdown("No selected best models were available for detailed analysis."))
    else:
        load_cache = {}
        for winner in winners.values():
            row = winner["row"]
            reasons = ", ".join(winner["reasons"])
            metadata = load_metadata_from_row(row)
            loaded = load_model_once(row, metadata, load_cache)
            loaded_model = loaded["model"]
            loading_error = loaded["error"]
            framework = str(row.get("framework", "")).lower()
            scope = row.get("stage") or metadata.get("fold") or "not available"
            model_name = row.get("best_model") or metadata.get("model_name") or metadata.get("model_id") or "not available"

            display(Markdown(f"### {row['framework']} | {row['config']} | {scope}"))
            display(
                Markdown(
                    f"- `model:` {model_name}  \n"
                    f"- `selected because:` {reasons}  \n"
                    f"- `performance:` {comparison_metric_text(row)}"
                )
            )

            if loading_error:
                display(
                    pd.DataFrame(
                        [
                            {
                                "status": "failed",
                                "artifact": compact_value(loaded["artifact_path"], 84),
                                "error": compact_error(loading_error),
                                "available spec": metadata.get("algo") or metadata.get("model_type") or "metadata only",
                            }
                        ]
                    )
                )
                continue

            if framework == "tpot":
                display(collect_pipeline_rows(loaded_model))
            elif framework == "h2o":
                display(extract_h2o_spec(loaded_model, metadata))
            elif framework == "autosklearn":
                display(extract_autosklearn_spec(loaded_model))
            else:
                display(collect_pipeline_rows(loaded_model))


### h2o | medium | final

- `model:` GBM_1_AutoML_6_20260425_151602  
- `selected because:` best overall performance, best configuration  
- `performance:` mean_rmse=53.6648; validation_score=28.2230

,algorithm,key hyperparameters
0,GBM,"ntrees=80, max_depth=15, sample_rate=0.8, col_sample_rate=0.8, col_sample_rate_per_tree=0.8"


### h2o | low | final

- `model:` GBM_1_AutoML_6_20260425_150556  
- `selected because:` fastest training time  
- `performance:` mean_rmse=58.7723; validation_score=28.7248

...

- `model:` Pipeline  
- `selected because:` lightest/simplest model  
- `performance:` mean_rmse=98.8364

,step,estimator,key hyperparameters
0,1.pipeline-1,Pipeline,not available
1,1.pipeline-1 > 1.impute_categorical,ColumnSimpleImputer,"columns=categorical, missing_values=nan, strategy=most_frequent"
2,1.pipeline-1 > 2.impute_numeric,ColumnSimpleImputer,"columns=numeric, missing_values=nan"
3,1.pipeline-1 > 3.ColumnOneHotEncoder,ColumnOneHotEncoder,"columns=categorical, min_frequency=0.0001"
4,2.pipeline-2,Pipeline,not available
5,2.pipeline-2 > 1.normalizer,Normalizer,norm=l1
6,...,truncated,intermediate nested steps omitted
7,2.pipeline-2 > 5.decisiontreeregressor,DecisionTreeRegressor [final],"min_samples_leaf=13, min_samples_split=9, random_state=42"


### tpot | medium | final

- `model:` Pipeline  
- `selected because:` most stable across folds  
- `performance:` mean_rmse=69.6984

,step,estimator,key hyperparameters
0,1.pipeline-1,Pipeline,not available
1,1.pipeline-1 > 1.impute_categorical,ColumnSimpleImputer,"columns=categorical, missing_values=nan, strategy=most_frequent"
2,1.pipeline-1 > 2.impute_numeric,ColumnSimpleImputer,"columns=numeric, missing_values=nan"
3,1.pipeline-1 > 3.ColumnOneHotEncoder,ColumnOneHotEncoder,"columns=categorical, min_frequency=0.0001"
4,2.pipeline-2,Pipeline,not available
5,2.pipeline-2 > 1.robustscaler,RobustScaler,"quantile_range=(0.2044303156983, 0.72702..."
6,...,truncated,intermediate nested steps omitted
7,2.pipeline-2 > 5.ridgecv,RidgeCV [final],not available
